# 14주차 — 함정·보안·비정형
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 함정 ① 단위 불일치

In [ ]:
# 혈당을 낮은 값부터 — 이 7.8은 저혈당인가?
q("""
SELECT lab_id, test_name, result_value, unit
FROM lab WHERE test_name = 'Glucose'
ORDER BY result_value LIMIT 5
""")

In [ ]:
# 단위 감사 쿼리 — 분석 전 30초 습관
q("""
SELECT test_name, unit, COUNT(*) AS 건수,
       MIN(result_value) AS 최소, MAX(result_value) AS 최대
FROM lab GROUP BY test_name, unit ORDER BY test_name
""")

## 함정 ② 분모 — '검사받은 사람만'의 평균

In [ ]:
# 당뇨 환자 수 vs HbA1c 검사자 수
q("""
SELECT
  (SELECT COUNT(DISTINCT v.patient_id) FROM visit v
   JOIN diagnosis d ON d.visit_id = v.visit_id
   WHERE d.kcd_code LIKE 'E11%') AS 당뇨환자수,
  (SELECT COUNT(DISTINCT v.patient_id) FROM visit v
   JOIN lab l ON l.visit_id = v.visit_id
   WHERE l.test_name = 'HbA1c') AS 검사받은환자수
""")

## 함정 ③ 비정형 텍스트의 한계

In [ ]:
# 판독문 LIKE 검색 — '경색'이 들어간 보고서
q("""
SELECT img_id, modality, body_part, report
FROM imaging WHERE report LIKE '%경색%'
""")

⚠️ "급성 경색 **소견 없음**" 같은 부정 표현도 걸립니다 — 텍스트의 의미는 LIKE가 이해하지 못합니다.

## 연습문제

In [ ]:
# [연습 Q1] mmol/L로 입력된 혈당을 mg/dL로 환산하면 얼마인가? (×18) — SELECT로 환산값을 계산해 보라
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 채혈시각(collected_at)의 시(hour) 분포를 구하라 — 오후 채혈이 섞여 있는가?
# 힌트: strftime('%H', collected_at)
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 각 검사(test_name)의 결측(result_value IS NULL) 건수를 세어 보라
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 판독문에 '없음'이 포함된 영상 보고서를 찾아, LIKE 검색의 함정을 확인하라
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 단위 환산**
```sql
SELECT lab_id, result_value AS mmol, ROUND(result_value * 18, 0) AS mgdl
FROM lab WHERE unit = 'mmol/L';
```

**A2. 채혈 시각 분포**
```sql
SELECT strftime('%H', collected_at) AS 시, COUNT(*) FROM lab GROUP BY 시 ORDER BY 시;
```

**A3. 검사별 결측**
```sql
SELECT test_name, SUM(CASE WHEN result_value IS NULL THEN 1 ELSE 0 END) AS 결측 FROM lab GROUP BY test_name;
```

**A4. 부정 표현**
```sql
SELECT img_id, report FROM imaging WHERE report LIKE '%없음%';
```